# Gold Layer 3 — Hospital Pressure & Operational Activity

**Goal:** Create a daily care-unit-level Gold dataset that summarises hospital activity and identifies periods of unusually high operational pressure.

**Grain:** One row per `activity_date + careunit`.

The dataset will combine admission/discharge activity, unit-level patient presence and transfers, and ICU activity. Any derived pressure indicator represents relative activity within the MIMIC-IV dataset and should not be interpreted as actual hospital capacity utilisation.

In [0]:
# Gold 3 source tables
silver_catalog = "cdr_silver.processed_mimic_iv"

source_tables = [
    "admissions",
    "transfers",
    "icustays",
    "services"
]

for table_name in source_tables:
    print(f"\n--- {table_name.upper()} ---")
    spark.table(f"{silver_catalog}.{table_name}").printSchema()

## 1. Validate Transfer Activity

`transfers` will be the main source for unit-level patient activity.  
Before expanding transfer intervals into daily care-unit records, we check event types, missing care units, missing admission IDs, and incomplete or invalid time ranges.

In [0]:
from pyspark.sql import functions as F

transfers = spark.table(
    "cdr_silver.processed_mimic_iv.transfers"
)

# Basic transfer quality checks
transfer_quality = transfers.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transfer_id").alias("unique_transfer_ids"),
    F.sum(F.col("hadm_id").isNull().cast("int")).alias("missing_hadm_id"),
    F.sum(F.col("careunit").isNull().cast("int")).alias("missing_careunit"),
    F.sum(F.col("intime").isNull().cast("int")).alias("missing_intime"),
    F.sum(F.col("outtime").isNull().cast("int")).alias("missing_outtime"),
    F.sum(
        (
            F.col("intime").isNotNull()
            & F.col("outtime").isNotNull()
            & (F.col("outtime") < F.col("intime"))
        ).cast("int")
    ).alias("outtime_before_intime")
)

display(transfer_quality)

# Event-type distribution
display(
    transfers
    .groupBy("eventtype")
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"))
)

# Most common care units, including missing values
display(
    transfers
    .groupBy("careunit")
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"))
)

In [0]:
# Check care-unit and timing quality by event type
display(
    transfers
    .groupBy("eventtype")
    .agg(
        F.count("*").alias("rows"),
        F.sum((F.col("careunit") == "UNKNOWN").cast("int")).alias("unknown_careunit"),
        F.sum(F.col("careunit").isNull().cast("int")).alias("null_careunit"),
        F.sum(F.col("intime").isNull().cast("int")).alias("missing_intime"),
        F.sum(F.col("outtime").isNull().cast("int")).alias("missing_outtime"),
        F.sum(
            (
                F.col("intime").isNotNull()
                & F.col("outtime").isNotNull()
                & (F.col("outtime") < F.col("intime"))
            ).cast("int")
        ).alias("invalid_time_order")
    )
    .orderBy("eventtype")
)

## 2. Review Incomplete Transfer Intervals

Most admission, transfer and ED records contain valid care-unit intervals.  
A small number contain missing or inconsistent end times, so these records are reviewed before defining the daily patient-load rules.

In [0]:
# Review incomplete or invalid care-unit intervals
problem_transfers = (
    transfers
    .filter(
        (F.col("eventtype").isin("ED", "admit", "transfer"))
        & (
            (F.col("careunit") == "UNKNOWN")
            | F.col("outtime").isNull()
            | (
                F.col("outtime").isNotNull()
                & (F.col("outtime") < F.col("intime"))
            )
        )
    )
    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "outtime"
    )
    .orderBy("eventtype", "hadm_id", "intime")
)

display(problem_transfers)

## 3. Assess Recoverable Missing Transfer End Times

A small number of admission-to-unit records have a known care unit and start time but no recorded end time.

We test whether these intervals can be completed using the next recorded hospital movement or, where no later movement exists, the hospital discharge time.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

admissions = spark.table(
    "cdr_silver.processed_mimic_iv.admissions"
)

# Order all transfer events within each hospital admission
transfer_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy("intime", "transfer_id")
)

transfer_sequence = (
    transfers
    .withColumn(
        "next_event_intime",
        F.lead("intime").over(transfer_window)
    )
)

# Known care-unit admissions where outtime is missing
missing_admit_outtime = (
    transfer_sequence
    .filter(
        (F.col("eventtype") == "admit")
        & (F.col("careunit") != "UNKNOWN")
        & F.col("outtime").isNull()
    )
    .join(
        admissions.select("hadm_id", "dischtime"),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "candidate_outtime",
        F.coalesce("next_event_intime", "dischtime")
    )
)

display(
    missing_admit_outtime
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("next_event_intime").isNotNull().cast("int"))
            .alias("recoverable_from_next_event"),
        F.sum(
            (
                F.col("next_event_intime").isNull()
                & F.col("dischtime").isNotNull()
            ).cast("int")
        ).alias("recoverable_from_discharge"),
        F.sum(F.col("candidate_outtime").isNull().cast("int"))
            .alias("still_unresolved")
    )
)

## 4. Build Valid Care-Unit Intervals

The inpatient activity backbone uses admission and transfer records with known care units.

Where an admission-to-unit record has a missing end time, the next recorded hospital movement is used as the inferred interval end. These inferred intervals are explicitly flagged.

In [0]:
# Build cleaned inpatient care-unit intervals
careunit_intervals = (
    transfer_sequence
    .filter(
        F.col("eventtype").isin("admit", "transfer")
        & (F.col("careunit") != "UNKNOWN")
        & F.col("hadm_id").isNotNull()
        & F.col("intime").isNotNull()
    )
    .withColumn(
        "outtime_inferred",
        F.col("outtime").isNull()
    )
    .withColumn(
        "effective_outtime",
        F.coalesce("outtime", "next_event_intime")
    )
    .filter(
        F.col("effective_outtime").isNotNull()
        & (F.col("effective_outtime") >= F.col("intime"))
    )
    .select(
        "subject_id",
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "effective_outtime",
        "outtime_inferred"
    )
)

# Validate the cleaned interval set
display(
    careunit_intervals.agg(
        F.count("*").alias("valid_intervals"),
        F.countDistinct("transfer_id").alias("unique_transfer_ids"),
        F.countDistinct("hadm_id").alias("distinct_admissions"),
        F.sum(F.col("outtime_inferred").cast("int")).alias("inferred_outtimes"),
        F.min("intime").alias("earliest_intime"),
        F.max("effective_outtime").alias("latest_outtime")
    )
)

## 5. Validate Care-Unit Interval Durations

Before expanding care-unit intervals into daily activity records, interval durations are checked for zero-length or unusually long stays that could distort daily patient-load calculations.

In [0]:
# Check care-unit interval durations
interval_check = (
    careunit_intervals
    .withColumn(
        "interval_hours",
        (F.col("effective_outtime").cast("long") - F.col("intime").cast("long")) / 3600
    )
)

display(
    interval_check.agg(
        F.count("*").alias("intervals"),
        F.sum((F.col("interval_hours") == 0).cast("int")).alias("zero_duration"),
        F.sum((F.col("interval_hours") > 24 * 30).cast("int")).alias("over_30_days"),
        F.sum((F.col("interval_hours") > 24 * 100).cast("int")).alias("over_100_days"),
        F.min("interval_hours").alias("min_hours"),
        F.expr("percentile_approx(interval_hours, 0.5)").alias("median_hours"),
        F.expr("percentile_approx(interval_hours, 0.95)").alias("p95_hours"),
        F.max("interval_hours").alias("max_hours")
    )
)

## 6. Review Extreme Care-Unit Intervals

Most care-unit intervals have plausible durations, but a small number are unusually long.

Intervals longer than 100 days and zero-duration intervals are reviewed against the parent hospital admission before daily activity records are generated.

In [0]:
# Add hospital admission boundaries for validation
interval_extremes = (
    interval_check
    .join(
        admissions.select(
            "hadm_id",
            F.col("admittime").alias("hospital_admittime"),
            F.col("dischtime").alias("hospital_dischtime")
        ),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "starts_before_admission",
        F.col("intime") < F.col("hospital_admittime")
    )
    .withColumn(
        "ends_after_discharge",
        F.col("effective_outtime") > F.col("hospital_dischtime")
    )
)

# Summary of the >100 day intervals
display(
    interval_extremes
    .filter(F.col("interval_hours") > 24 * 100)
    .agg(
        F.count("*").alias("over_100_day_intervals"),
        F.sum(F.col("starts_before_admission").cast("int"))
            .alias("start_before_admission"),
        F.sum(F.col("ends_after_discharge").cast("int"))
            .alias("end_after_discharge"),
        F.sum(F.col("outtime_inferred").cast("int"))
            .alias("inferred_outtime"),
        F.countDistinct("hadm_id").alias("affected_admissions")
    )
)

# Show the longest intervals
display(
    interval_extremes
    .filter(
        (F.col("interval_hours") > 24 * 100)
        | (F.col("interval_hours") == 0)
    )
    .select(
        "hadm_id",
        "transfer_id",
        "eventtype",
        "careunit",
        "intime",
        "effective_outtime",
        "interval_hours",
        "hospital_admittime",
        "hospital_dischtime",
        "starts_before_admission",
        "ends_after_discharge",
        "outtime_inferred"
    )
    .orderBy(F.desc("interval_hours"))
)

## 7. Validate Care-Unit Intervals Against Hospital Admission Bounds

Care-unit intervals should normally fall within the associated hospital admission.

Rather than removing unusually long stays based on duration alone, intervals are compared with the hospital admission and discharge timestamps so genuine long admissions can be retained while inconsistent transfer intervals can be corrected or excluded.

In [0]:
# Validate all care-unit intervals against admission boundaries
boundary_check = (
    interval_check
    .join(
        admissions.select(
            "hadm_id",
            F.col("admittime").alias("hospital_admittime"),
            F.col("dischtime").alias("hospital_dischtime")
        ),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "starts_before_admission",
        F.col("intime") < F.col("hospital_admittime")
    )
    .withColumn(
        "starts_after_discharge",
        F.col("intime") > F.col("hospital_dischtime")
    )
    .withColumn(
        "ends_before_admission",
        F.col("effective_outtime") < F.col("hospital_admittime")
    )
    .withColumn(
        "ends_after_discharge",
        F.col("effective_outtime") > F.col("hospital_dischtime")
    )
    .withColumn(
        "start_difference_minutes",
        (
            F.col("hospital_admittime").cast("long")
            - F.col("intime").cast("long")
        ) / 60
    )
    .withColumn(
        "end_difference_minutes",
        (
            F.col("effective_outtime").cast("long")
            - F.col("hospital_dischtime").cast("long")
        ) / 60
    )
)

display(
    boundary_check.agg(
        F.count("*").alias("intervals"),
        F.sum(F.col("starts_before_admission").cast("int"))
            .alias("starts_before_admission"),
        F.sum(F.col("starts_after_discharge").cast("int"))
            .alias("starts_after_discharge"),
        F.sum(F.col("ends_before_admission").cast("int"))
            .alias("ends_before_admission"),
        F.sum(F.col("ends_after_discharge").cast("int"))
            .alias("ends_after_discharge"),
        F.sum(
            (
                F.col("starts_after_discharge")
                | F.col("ends_before_admission")
            ).cast("int")
        ).alias("fully_outside_admission")
    )
)

## 7. Validate Care-Unit Intervals Against Hospital Admission Bounds

Care-unit intervals should normally fall within the associated hospital admission.

Rather than removing unusually long stays based on duration alone, intervals are compared with the hospital admission and discharge timestamps so genuine long admissions can be retained while inconsistent transfer intervals can be corrected or excluded.

In [0]:
# Validate all care-unit intervals against admission boundaries
boundary_check = (
    interval_check
    .join(
        admissions.select(
            "hadm_id",
            F.col("admittime").alias("hospital_admittime"),
            F.col("dischtime").alias("hospital_dischtime")
        ),
        on="hadm_id",
        how="left"
    )
    .withColumn(
        "starts_before_admission",
        F.col("intime") < F.col("hospital_admittime")
    )
    .withColumn(
        "starts_after_discharge",
        F.col("intime") > F.col("hospital_dischtime")
    )
    .withColumn(
        "ends_before_admission",
        F.col("effective_outtime") < F.col("hospital_admittime")
    )
    .withColumn(
        "ends_after_discharge",
        F.col("effective_outtime") > F.col("hospital_dischtime")
    )
    .withColumn(
        "start_difference_minutes",
        (
            F.col("hospital_admittime").cast("long")
            - F.col("intime").cast("long")
        ) / 60
    )
    .withColumn(
        "end_difference_minutes",
        (
            F.col("effective_outtime").cast("long")
            - F.col("hospital_dischtime").cast("long")
        ) / 60
    )
)

display(
    boundary_check.agg(
        F.count("*").alias("intervals"),
        F.sum(F.col("starts_before_admission").cast("int"))
            .alias("starts_before_admission"),
        F.sum(F.col("starts_after_discharge").cast("int"))
            .alias("starts_after_discharge"),
        F.sum(F.col("ends_before_admission").cast("int"))
            .alias("ends_before_admission"),
        F.sum(F.col("ends_after_discharge").cast("int"))
            .alias("ends_after_discharge"),
        F.sum(
            (
                F.col("starts_after_discharge")
                | F.col("ends_before_admission")
            ).cast("int")
        ).alias("fully_outside_admission")
    )
)